# 02 - Baselines (classic models, no Gemini)

Trains four non-Gemini classifiers that Gemini must beat: TF-IDF, embeddings + logistic regression,
embeddings + nearest neighbours, and a fine-tuned ModernBERT. **Cost: $0.**
**Needs a GPU:** Runtime > Change runtime type > T4 GPU, *before* running setup.
Takes roughly 45-90 minutes, mostly ModernBERT training. Trained models are saved on Drive, so a
disconnect only costs the step that was running.

**Step 1 - setup (free).** Downloads the project code from GitHub, installs it, connects Google Drive
(all results are saved in `MyDrive/cfpb-nlp-classifier/phase1`), reads your `GEMINI_API_KEY` secret (and the optional `TYPESAFE_API_KEY` Jev secret) and
logs in to Google Sheets. Colab will ask you to allow Drive and Sheets access - click Allow.
Takes 2-4 minutes the first time. If it says the key is missing: click the key icon on the left of
Colab, add a secret named `GEMINI_API_KEY`, and switch "Notebook access" on.
On the first run after a project update, setup may say it moved old results into an `archive_...`
folder on Drive. That is expected and needs nothing from you: your spending record, saved Gemini
answers and saved choices stay where they are.

In [ ]:
!git clone -q https://github.com/Dentoneer/cfpb-nlp-classifier /content/cfpb-nlp-classifier || git -C /content/cfpb-nlp-classifier pull -q
# Installs this project only; never upgrades Colab's own Google libraries.
%pip install -q -e "/content/cfpb-nlp-classifier[ml]"
import sys; sys.path.insert(0, "/content/cfpb-nlp-classifier")
from nlp.colab import setup
cfg, gc, api_key, jev_key = setup()

**Load the saved splits (free).** Builds the train frame, the tuning + test frames to predict, and the
test set to score.

In [ ]:
import pandas as pd
from nlp import colab, store, pipeline as P
complaints = store.read(cfg, P.STAGE_COMPLAINTS)
splits = store.read(cfg, P.STAGE_SPLITS)
train_df, test_df = P.train_test_frames(complaints, splits)
tuning_df = P.split_rows(complaints, splits, splits["split"] == "tuning")
eval_df = pd.concat([tuning_df, test_df], ignore_index=True)
print(len(train_df), "train,", len(tuning_df), "tuning,", len(test_df), "test")

**TF-IDF and the two embedding models.** TF-IDF is the simple word-count baseline Gemini has to beat.
The embedding models turn each complaint into numbers with a small neural net (uses the GPU).
**Look at:** the accuracy table in the next cell after all three finish.

In [ ]:
from nlp.classifiers.tfidf import TfidfLR
from nlp.classifiers.embed import EmbedLR, EmbedKNN
colab.run_classic(cfg, "tfidf", lambda: TfidfLR(), train_df, eval_df)
colab.run_classic(cfg, "embed_lr", lambda: EmbedLR(cfg.embed_model), train_df, eval_df)
colab.run_classic(cfg, "knn", lambda: EmbedKNN(cfg.embed_model), train_df, eval_df)

**ModernBERT (the long step).** Fine-tunes a transformer on the training complaints (about 30-60 minutes on
a T4). If Colab disconnects, reconnect, rerun setup and this cell: a finished model is loaded from Drive.

In [ ]:
from nlp.classifiers.modernbert import ModernBertClassifier
colab.run_classic(cfg, "modernbert", lambda: ModernBertClassifier(), train_df, eval_df)

**Baseline accuracy on the test set (free).** Share of test complaints where the model agrees with the
CFPB label, at each level. **Look at:** TF-IDF - Gemini has to beat it at every level. The 5-fold
out-of-fold TF-IDF probabilities for cleanlab are computed in notebook 04 (they need Gemini's
answers), also at no cost.

In [ ]:
rows = {n: colab.accuracy_by_level(store.read(cfg, P.stage_pred(n)), test_df)
        for n in ["tfidf", "embed_lr", "knn", "modernbert"]}
pd.DataFrame(rows).T.round(3)

In [ ]:
from nlp import colab
ledger, *_ = colab.open_tools(cfg)
print(colab.ledger_report(ledger))

Done. Next: `03_gemini.ipynb`. This is where money is spent.